In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [ ]:
with open("dataset.txt", "r", encoding="utf-8") as file:
    data = file.read()

tokenaization the text data

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([data])

sequences = tokenizer.texts_to_sequences([data])

print(f"number of unique tokens (word) : {len(tokenizer.word_index)}")
print("first 10 tokens:", list(tokenizer.word_index.items())[:10])
print("first sequence (fragment)", sequences[0][:20])

number of unique tokens (word) : 4993
first 10 tokens: [('the', 1), ('and', 2), ('a', 3), ('of', 4), ('to', 5), ('i', 6), ('you', 7), ('in', 8), ('is', 9), ('monica', 10)]
first sequence (fragment) [1, 155, 21, 2368, 1549, 8, 1, 422, 692, 215, 2, 3, 2369, 1550, 2370, 1, 423, 4, 1, 1142]


In [ ]:
input_sequences = []
for sentence in data.split('\n'):
  tokenized_sequence = tokenizer.texts_to_sequences([sentence])[0]

  for i in range(1,len(tokenized_sequence)):
    input_sequences.append(tokenized_sequence[:i+1])

print(f"total number of  input sequence : {len(input_sequences)}")
print("first 5 input sequences (before padding):")
for i in range(10):
  print(input_sequences[i])

total number of  input sequence : 26383
first 5 input sequences (before padding):
[1, 155]
[1, 155, 21]
[1, 155, 21, 2368]
[1, 155, 21, 2368, 1549]
[1, 155, 21, 2368, 1549, 8]
[1, 155, 21, 2368, 1549, 8, 1]
[1, 155, 21, 2368, 1549, 8, 1, 422]
[1, 155, 21, 2368, 1549, 8, 1, 422, 692]
[1, 155, 21, 2368, 1549, 8, 1, 422, 692, 215]
[1, 155, 21, 2368, 1549, 8, 1, 422, 692, 215, 2]


In [ ]:
max_len = max(len(x) for x in input_sequences)
max_len

325

In [ ]:
print(len(tokenizer.word_index))

4993


In [ ]:
padding_input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')

In [ ]:
x = padding_input_sequences[:,:-1]
x

array([[   0,    0,    0, ...,    0,    0,    1],
       [   0,    0,    0, ...,    0,    1,  155],
       [   0,    0,    0, ...,    1,  155,   21],
       ...,
       [   0,    0,    0, ...,   64, 2331,  290],
       [   0,    0,    0, ..., 2331,  290,   19],
       [   0,    0,    0, ...,  290,   19,   54]], dtype=int32)

In [ ]:
y = padding_input_sequences[:,-1]
y

array([ 155,   21, 2368, ...,   19,   54, 1535], dtype=int32)

In [ ]:
x.shape

(26383, 324)

In [ ]:
y.shape

(26383,)

In [ ]:
from tensorflow.keras.utils import to_categorical
# Assuming y contains your target labels
# Determine the maximum label value in your dataset
num_class = len(tokenizer.word_index) + 1  # Adding 1 to account for zero-based indexing

# Convert y to one-hot encoded format
y_encoded = to_categorical(y, num_classes=num_class)
print(y_encoded.shape)


(26383, 4994)


In [ ]:
print(y_encoded.shape)
y_encoded[0]

(26383, 4994)


array([0., 0., 0., ..., 0., 0., 0.])

In [ ]:
vocab_size = 4994
embedding_dim = 100

model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim))
model.add(LSTM(150))
model.add(Dense(vocab_size, activation="softmax"))

model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

# Print the model summary
model.summary()



Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.fit(x, y_encoded, epochs=100, verbose=2)

Epoch 1/100
825/825 - 22s - 26ms/step - accuracy: 0.0597 - loss: 7.0354
Epoch 2/100
825/825 - 14s - 17ms/step - accuracy: 0.0806 - loss: 6.3282
Epoch 3/100
825/825 - 14s - 17ms/step - accuracy: 0.1009 - loss: 5.8972
Epoch 4/100
825/825 - 14s - 17ms/step - accuracy: 0.1204 - loss: 5.5077
Epoch 5/100
825/825 - 14s - 17ms/step - accuracy: 0.1430 - loss: 5.1277
Epoch 6/100
825/825 - 14s - 17ms/step - accuracy: 0.1638 - loss: 4.7685
Epoch 7/100
825/825 - 14s - 17ms/step - accuracy: 0.1886 - loss: 4.4240
Epoch 8/100
825/825 - 14s - 17ms/step - accuracy: 0.2173 - loss: 4.0844
Epoch 9/100
825/825 - 14s - 17ms/step - accuracy: 0.2574 - loss: 3.7618
Epoch 10/100
825/825 - 14s - 17ms/step - accuracy: 0.3045 - loss: 3.4457
Epoch 11/100
825/825 - 14s - 17ms/step - accuracy: 0.3616 - loss: 3.1478
Epoch 12/100
825/825 - 14s - 17ms/step - accuracy: 0.4128 - loss: 2.8705
Epoch 13/100
825/825 - 14s - 17ms/step - accuracy: 0.4630 - loss: 2.6150
Epoch 14/100
825/825 - 14s - 17ms/step - accuracy: 0.5109 - 